# 11. Collective variables from a molecular dynamics trajectory

**Domain:** computational chemistry / statistical mechanics.

A simulation produces high-dimensional coordinates, but the interesting slow physics —
a conformational change, a barrier crossing — happens along a few **collective
variables**. Finding the right CV is the central problem of enhanced sampling.

**Setup.** A particle on a double-well potential in $x$, harmonically confined in $y$.
We then rotate the coordinates by 30° and bury them among three fast decoy degrees of
freedom, standing in for solvent or vibrational modes. The true CV is a linear
combination the search must find while ignoring the decoys.


## Setup


In [ ]:
%pip install -q "beamfeat[units]" pandas matplotlib seaborn scikit-learn


In [ ]:
import warnings
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from beamfeat import BeamFeatClassifier

warnings.filterwarnings("ignore", message=".*valid feature names.*")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

CSV_DIR = Path("csv")
CSV_DIR.mkdir(exist_ok=True)          # created on first run, reused after


def save_csv(frame, name):
    # Write to csv/<name> once, then read back so every run starts from disk.
    path = CSV_DIR / name
    if not path.exists():
        frame.to_csv(path, index=False)
        print(f"wrote  {path}  ({len(frame)} rows)")
    else:
        print(f"cached {path}")
    return pd.read_csv(path)


def raw_coef(m):
    # beamfeat standardises internally; rescale to original units.
    return m.coef_ / m.scaler_.scale_


def pct_err(est, true):
    return 100.0 * (est - true) / true

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(0)


## Running the dynamics

Overdamped (Brownian) Langevin integration:

$$x_{t+1} = x_t - \nabla U(x_t)\,\Delta t + \sqrt{2k_BT\,\Delta t}\;\xi_t$$

with $k_BT$ chosen so the barrier is crossed many times — we need both basins
populated.


In [ ]:
def simulate(nsteps=30_000, dt=5e-3, kT=0.30, omega=8.0):
    x, y = -1.0, 0.0
    X = np.empty(nsteps)
    Y = np.empty(nsteps)
    noise = np.sqrt(2 * kT * dt)
    for i in range(nsteps):
        x += -4 * x * (x * x - 1) * dt + noise * rng.normal()   # double well
        y += -omega * y * dt + noise * rng.normal()             # stiff spring
        X[i], Y[i] = x, y
    return X, Y


X_true, Y_true = simulate()
print(f"{len(X_true):,} steps, {int(np.sum(np.diff(np.sign(X_true)) != 0))} barrier crossings")


## Hiding the coordinate

Rotate by 30°, then append three fast decoys — uncorrelated noise carrying no slow
information.


In [ ]:
theta = np.deg2rad(30.0)
c, s = np.cos(theta), np.sin(theta)

traj = pd.DataFrame({
    "u": c * X_true - s * Y_true,        # rotated, observable
    "v": s * X_true + c * Y_true,
    "d1": rng.normal(size=len(X_true)),  # decoys
    "d2": rng.normal(size=len(X_true)),
    "d3": 0.5 * rng.normal(size=len(X_true)),
    "basin": (X_true > 0).astype(int),   # label any good CV must predict
})
traj = save_csv(traj, "md_trajectory.csv")

print(f"basin populations: {traj.basin.value_counts().to_dict()}")
traj.head()


### Look before you fit


In [ ]:
FEATS = ["u", "v", "d1", "d2", "d3"]

print("variance per column:")
print(traj[FEATS].var().round(4))
print("\ncorrelation with basin label:")
print(traj[FEATS].corrwith(traj.basin).round(4))


Note what the variance table says: `d1` and `d2` have **larger** variance than `u` and
`v`. Any method that ranks coordinates by variance will pick the decoys. Hold that
thought.


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4))

ax[0].plot(X_true[::20], lw=.5)
ax[0].set_xlabel("step / 20"); ax[0].set_ylabel("x (hidden true CV)")
ax[0].set_title("Hopping between two basins")

ax[1].hist(X_true, bins=80, color="steelblue")
ax[1].set_xlabel("x"); ax[1].set_ylabel("counts")
ax[1].set_title("Bimodal — two wells")

sc = ax[2].scatter(traj.u[::20], traj.v[::20], c=traj.basin[::20],
                   s=2, alpha=.3, cmap="coolwarm")
ax[2].set_xlabel("u (observed)"); ax[2].set_ylabel("v (observed)")
ax[2].set_title("Rotated: neither axis alone separates")

plt.tight_layout()
plt.show()


The third panel is the problem in one picture. In the observed coordinates the two
basins are separated along a diagonal, so neither `u` nor `v` alone does the job.


## Discovering the CV

Ask for the feature that best separates the basins. Successive MD frames are highly
correlated, so we thin the trajectory — treating correlated frames as independent
observations would inflate significance.


In [ ]:
thin = slice(None, None, 10)
F = traj[FEATS].values
basin = traj.basin.values

clf = BeamFeatClassifier(max_depth=2, beam_width=40,
                         random_state=0).fit(F[thin], basin[thin])

print(f"accuracy       : {clf.score(F[thin], basin[thin]):.4f}")
print(f"fdr_controlled_: {clf.fdr_controlled_}")
print("discovered features:")
for f in clf.formulas():
    print("   ", f)


## Did it ignore the decoys?


In [ ]:
used = sorted({i for f in clf.formulas() for i in range(5) if f"x{i}" in f})

print("column indices used :", used)
print("names               :", [FEATS[i] for i in used])
print("decoys (x2,x3,x4) used:", any(i >= 2 for i in used))


## The comparison that matters: PCA

PCA is the standard unsupervised way to find collective coordinates. It ranks
directions by **variance**, which is not the same thing as relevance — and here the
decoys have the most variance.


In [ ]:
pca = PCA(n_components=1).fit(F[thin])
pc1 = pca.transform(F[thin]).ravel()

print("PC1 loadings:")
for name, load in zip(FEATS, pca.components_[0]):
    print(f"   {name:<4} {load:+.3f}")

print(f"\n|corr(PC1, basin)|      = {abs(np.corrcoef(pc1, basin[thin])[0, 1]):.3f}")
print(f"|corr(true CV, basin)|  = {abs(np.corrcoef(X_true[thin], basin[thin])[0, 1]):.3f}")


PC1 loads almost entirely on `d1` and `d2` — the decoys — and correlates with the
basin label at essentially **zero**. It found the highest-variance direction, which
carries none of the slow physics.

This is the central lesson of the notebook. **Variance is not relevance.** An
unsupervised method that ranks by spread will happily hand you a fast, meaningless
coordinate. Supervised construction against a label the physics cares about does not
make that mistake.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].hist([pc1[basin[thin] == 0], pc1[basin[thin] == 1]],
           bins=50, label=["basin 0", "basin 1"], stacked=False)
ax[0].set_xlabel("PC1"); ax[0].set_title("PCA — basins fully overlap")
ax[0].legend()

cv = clf.decision_function(F[thin])
cv = cv[:, 0] if cv.ndim > 1 else cv
ax[1].hist([cv[basin[thin] == 0], cv[basin[thin] == 1]],
           bins=50, label=["basin 0", "basin 1"], stacked=False)
ax[1].set_xlabel("beamfeat CV"); ax[1].set_title("Discovered CV — bimodal, separated")
ax[1].legend()

plt.tight_layout()
plt.show()


## Baselines on the same task


In [ ]:
Ftr, Fte, btr, bte = train_test_split(F[thin], basin[thin],
                                      test_size=0.3, random_state=0, stratify=basin[thin])

logreg = make_pipeline(StandardScaler(),
                       LogisticRegression(max_iter=2000)).fit(Ftr, btr)
bf = BeamFeatClassifier(max_depth=2, beam_width=40, random_state=0).fit(Ftr, btr)

print(f"logistic regression (raw coords) : {logreg.score(Fte, bte):.4f}")
print(f"beamfeat classifier              : {bf.score(Fte, bte):.4f}")
print(f"majority class                   : {max(bte.mean(), 1 - bte.mean()):.4f}")


Logistic regression does well too — unsurprising, since the true CV is a *linear*
combination of `u` and `v`, which is exactly what logistic regression fits.

The difference is what you can carry forward. Logistic regression gives you five
coefficients on the raw coordinates. The constructed feature gives you an explicit
expression you can hand to an enhanced-sampling code as a biasing coordinate. For
umbrella sampling or metadynamics that difference is the whole point.


## Takeaways

1. The search recovered a CV separating the two basins from rotated coordinates,
   without being told which columns mattered.
2. It ignored the three decoys — check this explicitly, every time.
3. **PCA failed completely**, loading on the highest-variance directions, which were
   pure noise. Variance is not relevance.
4. Thin correlated trajectories before fitting. Successive MD frames are not
   independent observations.
5. Logistic regression matches on accuracy; the constructed feature wins on being an
   expression you can bias along.
